# Day 24: Model Comparison - Inference & Reasoning

Same puzzle. Different models. Different reasoning.

Today we compare Gemini Flash models on reasoning tasks:
- **Gemini 2.5 Flash-Lite** — Fastest, lightweight
- **Gemini 2.5 Flash** — Fast, multimodal (1M tokens)

## Setup

In [37]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv
import time

load_dotenv(dotenv_path='../.env')
API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)

In [38]:
def ask_model(model_name, prompt):
    """Ask a question to a specific model and return response with timing."""
    start = time.time()
    
    response = client.models.generate_content(
        model=model_name,
        contents=prompt
    )
    
    elapsed = time.time() - start
    return response.text, elapsed

from IPython.display import display, Markdown

def compare_models(prompt, models):
    """Compare multiple models on the same prompt."""
    display(Markdown(f"**Prompt:** {prompt[:100]}..."))
    print("=" * 60)
    
    for model in models:
        try:
            answer, time_taken = ask_model(model, prompt)
            display(Markdown(f"\n### 📌 {model} ({time_taken:.2f}s)"))
            display(Markdown(answer))
        except Exception as e:
            display(Markdown(f"\n### 📌 {model}"))
            print(f"Error: {e}")
        print("\n" + "-" * 60)

## Available Gemini Models

In [39]:
# Flash models only (free tier compatible)
MODELS = [
    "gemini-2.5-flash-lite",  # Fastest, lightweight
    "gemini-2.5-flash", # Fast, multimodal (1M tokens)
    "gemini-3-flash-preview",
]

print("Models we'll compare (Flash only):")
for m in MODELS:
    print(f"  - {m}")

Models we'll compare (Flash only):
  - gemini-2.5-flash-lite
  - gemini-2.5-flash
  - gemini-3-flash-preview


## Easy Puzzle: Probability

A simple probability question. Both models should get this right.

In [40]:
easy_puzzle = """You toss 2 coins. One of them is heads. 
What's the probability the other is tails? 
Answer with just the probability and brief explanation."""

compare_models(easy_puzzle, MODELS)

**Prompt:** You toss 2 coins. One of them is heads. 
What's the probability the other is tails? 
Answer with jus...


### 📌 gemini-2.5-flash-lite (1.39s)

1/2. Given that one coin is heads, there are two equally likely possibilities for the two coins: Heads-Tails or Tails-Heads. In one of these, the other coin is tails.


------------------------------------------------------------



### 📌 gemini-2.5-flash (4.61s)

2/3

**Explanation:** There are three equally likely outcomes where at least one coin is heads: (Heads, Heads), (Heads, Tails), (Tails, Heads). Of these three, two outcomes have the other coin as tails: (Heads, Tails) and (Tails, Heads).


------------------------------------------------------------



### 📌 gemini-3-flash-preview (2.83s)

**2/3**

There are four possible outcomes for two coins: HH, HT, TH, and TT. Knowing that at least one is heads eliminates TT, leaving three equally likely outcomes (HH, HT, TH). In two of those three cases (HT and TH), the other coin is tails.


------------------------------------------------------------


## Hard Puzzle: The Bookworm

A classic lateral thinking puzzle. This trips up many models!

In [41]:
hard_puzzle = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume 
to the last page of the second volume.
What distance did it gnaw through?
"""

compare_models(hard_puzzle, MODELS)

**Prompt:** 
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of e...


### 📌 gemini-2.5-flash-lite (2.74s)

Here's how to solve this problem:

**1. Understand the Setup**

* **Volume 1:** Pages (2 cm thick) + Cover 1 (2 mm thick) + Cover 2 (2 mm thick)
* **Volume 2:** Cover 1 (2 mm thick) + Cover 2 (2 mm thick) + Pages (2 cm thick)

The worm starts at the *first page* of the first volume and ends at the *last page* of the second volume.

**2. Visualize the Worm's Path**

Imagine the books on the shelf:

[Cover 1 (Vol 1)] [Pages (Vol 1)] [Cover 2 (Vol 1)] [Cover 1 (Vol 2)] [Pages (Vol 2)] [Cover 2 (Vol 2)]

The worm starts *inside* Cover 1 of Volume 1, on the first page. It travels through:

* The pages of Volume 1.
* The cover *between* Volume 1 and Volume 2 (this is the last cover of Volume 1 and the first cover of Volume 2).
* The pages of Volume 2.

**3. Identify What the Worm *Doesn't* Gnaw Through**

The worm starts *on* the first page of the first volume, so it doesn't gnaw through the very first cover of the first volume.

The worm stops *on* the last page of the second volume, so it doesn't gnaw through the very last cover of the second volume.

**4. Calculate the Distance**

The worm gnaws through:

* **Pages of Volume 1:** 2 cm
* **The cover between the volumes:** This is the last cover of Volume 1 AND the first cover of Volume 2. So, 2 mm + 2 mm = 4 mm
* **Pages of Volume 2:** 2 cm

**5. Convert Units to Be Consistent**

It's easiest to work with the same units. Let's convert centimeters to millimeters:

* 2 cm = 20 mm

**6. Add the Distances**

* Distance = (Pages of Vol 1) + (Cover between) + (Pages of Vol 2)
* Distance = 20 mm + 4 mm + 20 mm
* Distance = 44 mm

**Answer:** The worm gnawed through a distance of **44 mm** (or 4.4 cm).


------------------------------------------------------------



### 📌 gemini-2.5-flash (16.46s)

This is a classic riddle! The trick lies in visualizing how books are placed on a shelf.

1.  **Visualize the books on the shelf:**
    *   Volume 1 is on the left.
    *   Volume 2 is on the right.
    *   Their spines are facing outwards (away from the shelf).
    *   This means the **front cover** of Volume 1 is on its far left.
    *   The **back cover** of Volume 1 is on its far right, touching the **front cover** of Volume 2.
    *   The **back cover** of Volume 2 is on its far right.

    So, the order of parts from left to right is:
    [Front Cover V1] [Pages V1] [Back Cover V1] --- (touching) --- [Front Cover V2] [Pages V2] [Back Cover V2]

2.  **Trace the worm's path:**
    *   The worm starts "from the first page of the first volume." This means it *does not* gnaw through the front cover of Volume 1. It starts *after* it.
    *   It gnaws through **all the pages of Volume 1**. (2 cm)
    *   It gnaws through the **back cover of Volume 1**. (2 mm)
    *   It then reaches Volume 2. It gnaws through the **front cover of Volume 2**. (2 mm)
    *   It gnaws through **all the pages of Volume 2**. (2 cm)
    *   It ends "to the last page of the second volume." This means it *does not* gnaw through the back cover of Volume 2. It stops *before* it.

3.  **Calculate the total distance:**
    *   Pages of V1: 2 cm
    *   Back Cover of V1: 2 mm
    *   Front Cover of V2: 2 mm
    *   Pages of V2: 2 cm

    Convert all measurements to a single unit, for example, millimeters (mm):
    1 cm = 10 mm
    So, 2 cm = 20 mm

    Total distance = 20 mm (V1 pages) + 2 mm (V1 back cover) + 2 mm (V2 front cover) + 20 mm (V2 pages)
    Total distance = 44 mm

    Or, in centimeters: 4.4 cm

The worm gnawed through **44 mm** (or 4.4 cm).


------------------------------------------------------------



### 📌 gemini-3-flash-preview (80.85s)

To find the distance the worm gnawed, we need to look at how books are typically arranged on a bookshelf and where their pages are located.

### 1. The Arrangement of the Books
When two volumes (Volume 1 and Volume 2) stand side-by-side in the "usual" order (Volume 1 on the left, Volume 2 on the right) with their spines facing out:

*   **Volume 1 (on the left):** The front cover is on the **right** side of the book, and the back cover is on the **left**.
*   **Volume 2 (on the right):** The back cover is on the **left** side of the book, and the front cover is on the **right**.

You can verify this by looking at any book on a shelf: if the spine is facing you, the front cover is the one on the right. To open the book, you would pull the right-side cover toward you and swing it to the left.

### 2. The Location of the Pages
*   **The first page of Volume 1** is located immediately inside the **front cover**. Since the front cover of Volume 1 is on the right side of the book, the first page is at the very edge where it touches Volume 2.
*   **The last page of Volume 2** is located immediately inside the **back cover**. Since the back cover of Volume 2 is on the left side of the book, the last page is at the very edge where it touches Volume 1.

### 3. The Path of the Worm
The worm gnaws from the **first page of Volume 1** to the **last page of Volume 2**.
Based on the orientation above, the sequence of components from left to right is:
1.  Volume 1: [Back Cover] [All Pages] [**Front Cover**]
2.  Volume 2: [**Back Cover**] [All Pages] [Front Cover]

The worm starts at the first page of Volume 1 (which is at the boundary of the Volume 1 pages and its front cover) and gnaws through to the last page of Volume 2 (which is at the boundary of its back cover and the Volume 2 pages).

The worm's path consists only of:
*   The **front cover** of Volume 1 (2 mm)
*   The **back cover** of Volume 2 (2 mm)

### 4. Calculation
The worm does not gnaw through the 2 cm of pages in Volume 1, nor the 2 cm of pages in Volume 2, because the starting and ending points are at the edges of those page blocks.

$$\text{Distance} = \text{Front Cover thickness} + \text{Back Cover thickness}$$
$$\text{Distance} = 2\text{ mm} + 2\text{ mm} = 4\text{ mm}$$

**Answer:**
The worm gnawed through **4 mm** (or **0.4 cm**).


------------------------------------------------------------


### The Trick

When books are placed normally on a shelf:
- The **first page of Volume 1** is on the **right side** (facing Volume 2)
- The **last page of Volume 2** is on the **left side** (facing Volume 1)

So the worm only goes through:
- Front cover of Volume 1: 2mm
- Back cover of Volume 2: 2mm

**Answer: 4mm (or 0.4cm)**

Models that get this wrong typically assume the worm goes through all the pages.

## Math Puzzle: Step-by-Step Reasoning

In [ ]:
math_puzzle = """
A bat and ball cost $1.10 in total. 
The bat costs $1.00 more than the ball. 
How much does the ball cost?

Show your reasoning step by step.
"""

compare_models(math_puzzle, MODELS)

## Coding Puzzle: Find the Bug

In [ ]:
code_puzzle = """
What's wrong with this code?

```python
def find_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)

result = find_average([])
print(result)
```

Identify the bug and provide a fix. Don't give the corrected code, just answer in 2 lines
"""

compare_models(code_puzzle, MODELS)

## Speed vs Quality Trade-off

In [ ]:
simple_prompt = "What is the capital of France? Answer in one word."

print("Speed comparison on simple prompt:")
print("=" * 40)

for model in MODELS:
    try:
        answer, elapsed = ask_model(model, simple_prompt)
        print(f"{model}: {elapsed:.3f}s → {answer.strip()}")
    except Exception as e:
        print(f"{model}: Error - {e}")

## Key Takeaways

| Model | Speed | Reasoning | Best For |
|-------|-------|-----------|----------|
| **gemini-2.5-flash-lite** | ⚡⚡ Fastest | Good | Simple tasks, high volume |
| **gemini-2.5-flash** | ⚡ Fast | Better | Most tasks, multimodal |
| **gemini-3-flash-preview** | 🧪 Preview | Best | Latest features, testing |

**When to use which:**
- **Flash-Lite**: Maximum speed, simple prompts, high volume
- **Flash**: Default choice, better reasoning, multimodal support
- **3-Flash-Preview**: Bleeding edge, test new capabilities

**Key difference:** Flash-Lite is faster but may give shorter/simpler answers. Flash has better reasoning on complex puzzles. Gemini 3 brings latest improvements.

---

**That's Day 24!** Different models for different needs.

**Series complete!** Next: LangChain for more advanced workflows.